In [2]:
import numpy as np
import matplotlib.pyplot as plt
import re
import json
import pandas as pd


In [13]:
# Load the local Excel file
file_path = 'Drug parameters.xlsx'
# Skip the first two header rows
df = pd.read_excel(file_path, sheet_name='IC50', skiprows=2, header=None)

# Channel mapping for columns C through I
channels = ['INa', 'IKr', 'ICaL', 'INaL', 'IKs', 'Ito', 'IK1']

# Regex for IC50 and Hill coefficient: matches "Value(Hill)"
ic50_pattern = re.compile(r"([0-9.]+)\(([0-9.]+)\)")

# Regex for EFTPCmax: extracts the first numeric sequence (e.g., from "0.155#")
eftp_pattern = re.compile(r"([0-9.]+)")

drug_dict = {}

for index, row in df.iterrows():
    # Column B (index 1) is the Drug Name
    drug_name = str(row[1]).strip()
    
    # Skip empty rows
    if drug_name == 'nan' or not drug_name:
        continue
        
    drug_dict[drug_name] = {}

    # 1. Extract IC50 and Hill coefficient for each channel (Columns C-I)
    for i, channel in enumerate(channels):
        col_idx = i + 2
        cell_value = str(row[col_idx])
        
        if cell_value != 'nan' and cell_value != 'None' and cell_value.strip():
            match = ic50_pattern.search(cell_value)
            if match:
                drug_dict[drug_name][channel] = {
                    "IC50": float(match.group(1)),
                    "h": float(match.group(2))
                }
            else:
                # Fallback for IC50 only
                val_match = eftp_pattern.search(cell_value)
                if val_match:
                    drug_dict[drug_name][channel] = {
                        "IC50": float(val_match.group(1)),
                        "h": None
                    }

    # 2. Extract EFTPCmax (Column J / Index 9)
    eftp_cell = str(row[9])
    if eftp_cell != 'nan' and eftp_cell.strip():
        eftp_match = eftp_pattern.search(eftp_cell)
        if eftp_match:
            drug_dict[drug_name]['EFTPCmax'] = float(eftp_match.group(1))
        else:
            drug_dict[drug_name]['EFTPCmax'] = None
    else:
        drug_dict[drug_name]['EFTPCmax'] = None

# --- Usage Example ---
drug = "Amiodarone I"
if drug in drug_dict:
    data = drug_dict[drug]
    print(f"--- {drug} Parameters ---")
    print(f"EFTPCmax: {data['EFTPCmax']} µM")
    print(f"IKr IC50: {data.get('IKr', {}).get('IC50')} µM")
    print(f"IKr Hill (h): {data.get('IKr', {}).get('h')}")

--- Amiodarone I Parameters ---
EFTPCmax: 0.155 µM
IKr IC50: 0.86 µM
IKr Hill (h): 1.09


In [14]:
drug_dict['Amiodarone I'] 

{'INa': {'IC50': 15.9, 'h': 0.97},
 'IKr': {'IC50': 0.86, 'h': 1.09},
 'ICaL': {'IC50': 1.9, 'h': 0.69},
 'EFTPCmax': 0.155}